# 📡 Raspberry Pi 4B × Nexmon CSI 解析・学習ノートブック（Google Colab用）

このノートブックは、Raspberry Pi 4B が Google Drive（`マイドライブ/rpi4_nexmon_csi`）に自動で溜めている **Wi-Fi CSI 電波データ（`.pcap`）** と **正解ラベル（`segments.csv` / `labels.csv`）** を読み込み、可視化・前処理・お試し AI 分類まで行うためのものです。

### はじめて使うときのポイント
- **上から順番に ▶（再生ボタン）を押すだけ**で動きます。
- **ランタイムについて**:
  - グラフ確認・増分変換だけなら `CPU` ランタイムで十分（コンピューティングユニットを節約できます）。
  - 将来ディープラーニング（PyTorch CNN 等）を本格的に回すときだけ、メニューの `ランタイム > ランタイムのタイプを変更` から **L4 / T4 / A100 GPU** に切り替えてください。
- **データが消えない仕組み**:
  - 変換後の軽量ファイル（`.npz`）はすべて **Google Drive の `rpi4_nexmon_csi/processed/`** に保存されます。ランタイム接続が切れても、次回は変換済みファイルを自動でスキップして続きから再開できます。

## ステップ 1：Google Drive を接続（マウント）する
実行すると「Google ドライブへのアクセスを許可しますか？」というポップアップが出るので、**5TB の Google アカウント**を選んで許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/rpi4_nexmon_csi')
PROCESSED_DIR = ROOT / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Google Drive フォルダ確認:', ROOT.exists())
print('中身一覧:', [p.name for p in sorted(ROOT.iterdir())])

## ステップ 2：これまで集まったデータ量と 4 クラスの内訳を見る
`segments.csv`（5分ごとの `.pcap` と状態の対応表）と `labels.csv`（スマホでボタンを押した履歴）を読み込みます。

In [ ]:
segments = pd.read_csv(ROOT / 'segments.csv')
labels = pd.read_csv(ROOT / 'labels.csv')

total_gb = segments['bytes'].sum() / (1024 ** 3)
total_hrs = segments['duration_sec'].sum() / 3600
print(f'総セグメント数: {len(segments)} ファイル / 合計サイズ: {total_gb:.2f} GB / 合計時間: {total_hrs:.2f} 時間')
print()
print('--- 日付ごとの集計 ---')
display(segments.groupby('date').agg(files=('segment_id', 'count'), hours=('duration_sec', lambda s: round(s.sum()/3600, 2)), gb=('bytes', lambda s: round(s.sum()/(1024**3), 2))))

print('--- 4クラス（在室 × 窓開閉）ごとの集計 ---')
cls_summary = segments.groupby('label_class').agg(files=('segment_id', 'count'), hours=('duration_sec', lambda s: round(s.sum()/3600, 2)))
display(cls_summary)

ax = cls_summary['hours'].plot(kind='bar', figsize=(7, 3.5), color=['#60a5fa', '#a78bfa', '#34d399', '#fbbf24'])
ax.set_ylabel('Hours')
ax.set_title('Collected Hours per Class')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('--- スマホからの最新ラベル操作履歴 (直近10件) ---')
display(labels.tail(10))

## ステップ 3：`.pcap` ファイルを高速に読み込む関数を定義する
追加の C コンパイルや外部ライブラリ（`csiread` 等）に依存せず、Python 標準 + NumPy だけで **Raspberry Pi 4B (`BCM43455c0` / 80MHz / 256 サブキャリア)** の Nexmon CSI パケットを直接読み出します。

- 1 パケットの中身：`Ethernet(14B) + IPv4(20B) + UDP(8B) + Nexmonヘッダ(18B, Magic=0x1111) + CSIデータ(256 × 4B = 1024B)`
- あなたの環境では、5GHz 中継器（`ec:5a:31:a6:01:ce`）や親ルーター（`98:f1:99:a4:02:0f`）の Ch36 制御チャネル信号が、生 FFT インデックス **`134..191`（58 本のサブキャリア）** に非常に強い S/N 比で乗っていることを実機確認済みです。

In [ ]:
# あなたの自宅環境で実測確認済みの主な 5GHz 送信元 MAC アドレス
EXTENDER_MAC = 'ec:5a:31:a6:01:ce'  # Extender-5G-01C0 (平均 -48 dBm, 約 9.4 pkt/s)
ROUTER_MAC   = '98:f1:99:a4:02:0f'  # 親ルーター 5GHz (平均 -67 dBm, 約 27.3 pkt/s)

# Ch 36 (20MHz 制御チャネル帯域) に対応する生 FFT インデックス (ガード/DCヌル除外済み)
ACTIVE_BINS_Q2 = np.arange(134, 192)

def read_nexmon_pcap(pcap_path: Path, target_mac: str = None, max_packets: int = None):
    """
    Nexmon CSI (BCM43455c0, 80MHz, 256 subcarriers) の .pcap を読み込む。
    Returns: dict(timestamps, rssi, src_macs, csi_complex)
    """
    raw = Path(pcap_path).read_bytes()
    if len(raw) < 24:
        return None
    pos = 24  # PCAP グローバルヘッダ (24B)
    target_bytes = bytes.fromhex(target_mac.replace(':', '')) if target_mac else None

    ts_list = []
    rssi_list = []
    mac_list = []
    csi_chunks = []

    n_raw = len(raw)
    while pos + 16 <= n_raw:
        ts_sec, ts_usec, incl_len, _ = struct.unpack_from('<IIII', raw, pos)
        pos += 16
        if pos + incl_len > n_raw:
            break
        if incl_len >= 42 + 18 + 1024:
            udp_off = pos + 42
            magic = struct.unpack_from('<H', raw, udp_off)[0]
            if magic == 0x1111:
                mac_bytes = raw[udp_off + 4 : udp_off + 10]
                if target_bytes is None or mac_bytes == target_bytes:
                    rssi = struct.unpack_from('<b', raw, udp_off + 2)[0]
                    ts_list.append(ts_sec + ts_usec * 1e-6)
                    rssi_list.append(rssi)
                    if target_bytes is None:
                        mac_list.append(':'.join(f'{b:02x}' for b in mac_bytes))
                    csi_chunks.append(raw[udp_off + 18 : udp_off + 18 + 1024])
                    if max_packets is not None and len(ts_list) >= max_packets:
                        break
        pos += incl_len

    if not csi_chunks:
        return {'timestamps': np.array([]), 'rssi': np.array([]), 'src_macs': [], 'csi': np.empty((0, 256), dtype=np.complex64)}

    iq = np.frombuffer(b''.join(csi_chunks), dtype='<i2').reshape(-1, 256, 2)
    csi = (iq[:, :, 0].astype(np.float32) + 1j * iq[:, :, 1].astype(np.float32))
    return {
        'timestamps': np.array(ts_list, dtype=np.float64),
        'rssi': np.array(rssi_list, dtype=np.int16),
        'src_macs': mac_list,
        'csi': csi,
    }

print('read_nexmon_pcap 関数の準備完了！')

## ステップ 4：最新の `.pcap` を 1 本開いて電波のヒートマップを見てみる
横軸が「時間（パケット番号）」、縦軸が「周波数（サブキャリア番号）」、色が「電波の強さ（振幅）」です。部屋の中で動きがあると、縦の縞模様や色の揺らぎとして現れます。

In [ ]:
# Google Drive 上に確実に存在する最新の .pcap を後ろから探す
# (最新の1本はちょうど Pi からアップロード中・Drive反映待ちの場合があるため)
latest_row = None
latest_pcap = None
for _, r in segments.iloc[::-1].iterrows():
    cand = ROOT / r['relative_path']
    if cand.exists():
        latest_row = r
        latest_pcap = cand
        break

if latest_pcap is None:
    raise FileNotFoundError('Google Drive 上に .pcap が見つかりません。ステップ 1 を実行して Drive をマウントしてください。')

print('読み込み中:', latest_pcap.name, '| クラス:', latest_row['label_class'])

# 中継器 (EXTENDER_MAC) のパケットだけを抽出
d = read_nexmon_pcap(latest_pcap, target_mac=EXTENDER_MAC)
if len(d['timestamps']) == 0:
    print('指定 MAC が見つからなかったため、全パケットで読み込みます')
    d = read_nexmon_pcap(latest_pcap, max_packets=3000)

amp_active = np.abs(d['csi'][:, ACTIVE_BINS_Q2])
print(f'抽出パケット数: {len(d["timestamps"])} 件 / 平均 RSSI: {d["rssi"].mean():.1f} dBm')

plt.figure(figsize=(13, 4))
plt.imshow(amp_active.T, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='CSI Amplitude')
plt.xlabel('Packet Index (Time ->)')
plt.ylabel('Active Subcarrier Bin (Q2: 134..191)')
plt.title(f"{latest_row['filename']}  |  class={latest_row['label_class']}  |  MAC={EXTENDER_MAC}")
plt.tight_layout()
plt.show()

## ステップ 5：増分変換（重い `.pcap` → 軽い `.npz` にして Google Drive へ永続保存）
1 本 30MB〜70MB の生 `.pcap` を毎回読み直すと時間がかかるため、**必要な 2 台の送信元（中継器＋親ルーター）の有効サブキャリア振幅だけを取り出して 1 本数百 KB の `.npz` に圧縮保存**します。

- **すでに変換済みのファイルは自動スキップ**されるので、ランタイムが切れても前回の続きからすぐ終わります。
- まずはお試しとして `MAX_FILES_THIS_RUN = 40`（各クラスからバランスよく最大 40 本）に設定しています。全ファイルを変換したいときは `None` に変えて実行してください。

In [ ]:
MAX_FILES_THIS_RUN = 40  # 全件変換したいときは None に変更

# 各クラスから均等に未変換ファイルをピックアップ
todo_rows = []
for cls_name, grp in segments.groupby('label_class'):
    unconverted = [r for _, r in grp.iterrows() if not (PROCESSED_DIR / f"{r['segment_id']}.npz").exists()]
    limit = (MAX_FILES_THIS_RUN // max(1, segments['label_class'].nunique())) if MAX_FILES_THIS_RUN else len(unconverted)
    todo_rows.extend(unconverted[:limit])

print(f'今回の変換対象: {len(todo_rows)} ファイル (変換済みはスキップ)')

converted = 0
for i, row in enumerate(todo_rows, 1):
    pcap_path = ROOT / row['relative_path']
    out_path = PROCESSED_DIR / f"{row['segment_id']}.npz"
    if not pcap_path.exists():
        continue
    d_ext = read_nexmon_pcap(pcap_path, target_mac=EXTENDER_MAC)
    d_rtr = read_nexmon_pcap(pcap_path, target_mac=ROUTER_MAC)
    np.savez_compressed(
        out_path,
        segment_id=row['segment_id'],
        label_class=row['label_class'],
        presence=row['presence'],
        window=row['window'],
        ext_ts=d_ext['timestamps'],
        ext_rssi=d_ext['rssi'],
        ext_amp=np.abs(d_ext['csi'][:, ACTIVE_BINS_Q2]).astype(np.float32),
        rtr_ts=d_rtr['timestamps'],
        rtr_rssi=d_rtr['rssi'],
        rtr_amp=np.abs(d_rtr['csi'][:, ACTIVE_BINS_Q2]).astype(np.float32),
    )
    converted += 1
    if i % 10 == 0 or i == len(todo_rows):
        print(f'  [{i}/{len(todo_rows)}] 保存完了: {out_path.name}')

all_npz = sorted(PROCESSED_DIR.glob('*.npz'))
print(f'\nGoogle Drive 上の変換済み .npz 合計: {len(all_npz)} ファイル')

## ステップ 6：4 クラス（在室/不在 × 窓開/閉）の CSI 波形を並べて比較する
変換済みの `.npz` を使って、「部屋にいる／いない」「窓が開いている／閉まっている」で 58 本のサブキャリアの波形がどう違うかをグラフで比較します。

In [ ]:
class_profiles = {}
for npz_path in sorted(PROCESSED_DIR.glob('*.npz')):
    z = np.load(npz_path, allow_pickle=True)
    cls = str(z['label_class'])
    amp = z['ext_amp']
    if len(amp) < 50:
        continue
    # パケットごとに平均=1に正規化して波形の「形」を比較しやすくする
    norm = amp / (amp.mean(axis=1, keepdims=True) + 1e-6)
    class_profiles.setdefault(cls, []).append(norm.mean(axis=0))

plt.figure(figsize=(11, 4.5))
for cls, arrs in sorted(class_profiles.items()):
    mean_curve = np.mean(arrs, axis=0)
    plt.plot(ACTIVE_BINS_Q2, mean_curve, linewidth=2, label=f'{cls} (n={len(arrs)} files)')

plt.xlabel('Subcarrier Index (Ch36 Q2: 134..191)')
plt.ylabel('Normalized Mean CSI Amplitude')
plt.title('4 Classes Comparison: Average CSI Subcarrier Shape (Extender AP)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ステップ 7：お試し AI 分類（ベースライン機械学習モデル）
変換済みの `.npz` を **10 秒ごとの窓（ウィンドウ）** に区切り、各サブキャリアの平均・標準偏差（ゆらぎの大きさ）から 4 クラスを予測する簡易 AI モデル（RandomForest）を試します。

> **重要（リーク防止）**: 同じ 5 分 `.pcap` ファイル内のデータを学習用とテスト用に混ぜると精度が不当に高く見えてしまうため、**ファイル（`segment_id`）単位で学習用 (75%) とテスト用 (25%) に分けて** 正しく評価しています。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

WINDOW_SEC = 10.0
X_list, y_list, groups = [], [], []

for npz_path in sorted(PROCESSED_DIR.glob('*.npz')):
    z = np.load(npz_path, allow_pickle=True)
    cls = str(z['label_class'])
    seg_id = str(z['segment_id'])
    ts = z['ext_ts']
    amp = z['ext_amp']
    if len(ts) < 30:
        continue
    t0 = ts[0]
    win_ids = ((ts - t0) // WINDOW_SEC).astype(int)
    for w in np.unique(win_ids):
        mask = (win_ids == w)
        if mask.sum() < 20:
            continue
        sub = amp[mask]
        # 各サブキャリアの平均 + 標準偏差 + RSSI平均を特徴量にする
        feat = np.concatenate([
            sub.mean(axis=0),
            sub.std(axis=0),
            [z['ext_rssi'][mask].mean()],
        ])
        X_list.append(feat)
        y_list.append(cls)
        groups.append(seg_id)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)
groups = np.array(groups)
print(f'作成された 10秒サンプル数: {len(X)} 件 / ユニークなファイル数: {len(np.unique(groups))} 本')

if len(np.unique(y)) >= 2 and len(np.unique(groups)) >= 4:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X[train_idx], y[train_idx])
    pred = clf.predict(X[test_idx])

    print(classification_report(y[test_idx], pred))
    labels_sorted = sorted(np.unique(y))
    cm = confusion_matrix(y[test_idx], pred, labels=labels_sorted)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted)
    disp.plot(xticks_rotation=25, cmap='Blues')
    plt.title('Baseline RandomForest (File-Grouped Test Split)')
    plt.tight_layout()
    plt.show()
else:
    print('2クラス以上・4ファイル以上の変換済み .npz が揃うと分類テストを実行できます。')